In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
from tqdm import tqdm
import csv
from torchvision.models.segmentation import deeplabv3_resnet50
import torch.nn.functional as F


class LIDCSliceDataset(Dataset):
    
    def __init__(self, data_dir='lidc_processed', transform=None):
        self.data_dir = data_dir
        self.transform = transform
        
        self.image_dir = os.path.join(data_dir, 'images')
        self.mask_dir = os.path.join(data_dir, 'masks')
        
        self.sample_ids = [
            f.replace('.npy', '') 
            for f in os.listdir(self.image_dir) 
            if f.endswith('.npy')
        ]
        
        print(f"Found {len(self.sample_ids)} slices")
    
    def __len__(self):
        return len(self.sample_ids)
    
    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        
        image = np.load(os.path.join(self.image_dir, f'{sample_id}.npy'))
        mask = np.load(os.path.join(self.mask_dir, f'{sample_id}.npy'))
        
        if len(image.shape) == 2:
            image = np.stack([image, image, image], axis=0)
        else:
            image = np.transpose(image, (2, 0, 1))
            if image.shape[0] == 1:
                image = np.repeat(image, 3, axis=0)
        
        if len(mask.shape) == 2:
            mask = mask
        else:
            mask = mask[..., 0]
        
        image = torch.from_numpy(image).float()
        mask = torch.from_numpy(mask).long()
        
        return {
            'image': image,
            'mask': mask,
            'id': sample_id
        }


class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred = pred.contiguous().view(-1)
        target = target.contiguous().view(-1)
        
        intersection = (pred * target).sum()
        dice = (2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)
        
        return 1 - dice


def calculate_dice_score(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    
    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)
    
    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    
    return dice.item()


class DeepLabV3Segmentation:
    def __init__(self, num_classes=2, device='cuda', learning_rate=1e-4):
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.num_classes = num_classes
        
        self.model = deeplabv3_resnet50(pretrained=True, progress=True)
        self.model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
        self.model.aux_classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
        self.model = self.model.to(self.device)
        
        self.criterion = DiceLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)
        
        os.makedirs('checkpoints', exist_ok=True)
        os.makedirs('metrics', exist_ok=True)
    
    def train_epoch(self, dataloader):
        self.model.train()
        epoch_loss = 0.0
        epoch_dice = 0.0
        num_batches = 0
        
        progress_bar = tqdm(dataloader, desc='Training')
        
        for batch in progress_bar:
            images = batch['image'].to(self.device)
            masks = batch['mask'].to(self.device)
            
            self.optimizer.zero_grad()
            
            outputs = self.model(images)
            logits = outputs['out']
            
            logits_class1 = logits[:, 1:2, :, :]
            
            loss = self.criterion(logits_class1, masks.float())
            
            loss.backward()
            self.optimizer.step()
            
            dice = calculate_dice_score(logits_class1, masks.float())
            
            epoch_loss += loss.item()
            epoch_dice += dice
            num_batches += 1
            
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'dice': f'{dice:.4f}'
            })
        
        return epoch_loss / num_batches, epoch_dice / num_batches
    
    def validate(self, dataloader):
        self.model.eval()
        val_loss = 0.0
        val_dice = 0.0
        num_batches = 0
        
        with torch.no_grad():
            progress_bar = tqdm(dataloader, desc='Validation')
            
            for batch in progress_bar:
                images = batch['image'].to(self.device)
                masks = batch['mask'].to(self.device)
                
                outputs = self.model(images)
                logits = outputs['out']
                
                logits_class1 = logits[:, 1:2, :, :]
                
                loss = self.criterion(logits_class1, masks.float())
                dice = calculate_dice_score(logits_class1, masks.float())
                
                val_loss += loss.item()
                val_dice += dice
                num_batches += 1
                
                progress_bar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'dice': f'{dice:.4f}'
                })
        
        return val_loss / num_batches, val_dice / num_batches
    
    def save_checkpoint(self, epoch, train_loss, train_dice, val_loss, val_dice):
        checkpoint_path = os.path.join('checkpoints', f'deeplabv3_epoch_{epoch}.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'train_loss': train_loss,
            'train_dice': train_dice,
            'val_loss': val_loss,
            'val_dice': val_dice
        }, checkpoint_path)
        
        return checkpoint_path
    
    def save_metrics(self, epoch, train_loss, train_dice, val_loss, val_dice, checkpoint_path):
        csv_path = os.path.join('metrics', f'epoch_{epoch}_metrics.csv')
        
        with open(csv_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Metric', 'Value'])
            writer.writerow(['Epoch', epoch])
            writer.writerow(['Train Loss', train_loss])
            writer.writerow(['Train DICE', train_dice])
            writer.writerow(['Validation Loss', val_loss])
            writer.writerow(['Validation DICE', val_dice])
            writer.writerow(['Checkpoint Path', checkpoint_path])
            writer.writerow(['Model', 'DeepLabV3-ResNet50'])
            writer.writerow(['Loss Function', 'Dice Loss'])
            writer.writerow(['Optimizer', 'Adam'])
        
        return csv_path
    
    def train(self, train_loader, val_loader, num_epochs):
        print(f"Training on device: {self.device}")
        print(f"Model: DeepLabV3-ResNet50")
        print(f"Loss: Dice Loss")
        print(f"Optimizer: Adam")
        print(f"Number of epochs: {num_epochs}")
        print("-" * 50)
        
        for epoch in range(1, num_epochs + 1):
            print(f"\nEpoch {epoch}/{num_epochs}")
            
            train_loss, train_dice = self.train_epoch(train_loader)
            val_loss, val_dice = self.validate(val_loader)
            
            print(f"\nEpoch {epoch} Summary:")
            print(f"Train Loss: {train_loss:.4f}, Train DICE: {train_dice:.4f}")
            print(f"Val Loss: {val_loss:.4f}, Val DICE: {val_dice:.4f}")
            
            checkpoint_path = self.save_checkpoint(epoch, train_loss, train_dice, val_loss, val_dice)
            csv_path = self.save_metrics(epoch, train_loss, train_dice, val_loss, val_dice, checkpoint_path)
            
            print(f"Saved checkpoint: {checkpoint_path}")
            print(f"Saved metrics: {csv_path}")
            print("-" * 50)



dataset = LIDCSliceDataset('lidc_processed')

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

segmentation = DeepLabV3Segmentation(num_classes=2, device='cuda', learning_rate=1e-4)

segmentation.train(train_loader, val_loader, num_epochs=50)

Found 13677 slices
Train samples: 10941
Validation samples: 2736


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet50_coco-cd0a2569.pth
100%|██████████| 161M/161M [00:07<00:00, 22.1MB/s] 


Training on device: cuda
Model: DeepLabV3-ResNet50
Loss: Dice Loss
Optimizer: Adam
Number of epochs: 50
--------------------------------------------------

Epoch 1/50


Training:   1%|          | 33/2736 [00:10<13:57,  3.23it/s, loss=0.9948, dice=0.0089]

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
from tqdm import tqdm
import csv
from torchvision.models.segmentation import deeplabv3_resnet50
import torch.nn.functional as F


class LIDCSliceDataset(Dataset):
    
    def __init__(self, data_dir='lidc_processed', transform=None, preload=True):
        self.data_dir = data_dir
        self.transform = transform
        self.preload = preload
        
        self.image_dir = os.path.join(data_dir, 'images')
        self.mask_dir = os.path.join(data_dir, 'masks')
        
        self.sample_ids = [
            f.replace('.npy', '') 
            for f in os.listdir(self.image_dir) 
            if f.endswith('.npy')
        ]
        
        print(f"Found {len(self.sample_ids)} slices")
        
        if self.preload:
            print("Preloading dataset into RAM...")
            self.images = {}
            self.masks = {}
            
            for sample_id in tqdm(self.sample_ids, desc="Loading data"):
                image = np.load(os.path.join(self.image_dir, f'{sample_id}.npy'))
                mask = np.load(os.path.join(self.mask_dir, f'{sample_id}.npy'))
                
                if len(image.shape) == 2:
                    image = np.stack([image, image, image], axis=0)
                else:
                    image = np.transpose(image, (2, 0, 1))
                    if image.shape[0] == 1:
                        image = np.repeat(image, 3, axis=0)
                
                if len(mask.shape) == 2:
                    mask = mask
                else:
                    mask = mask[..., 0]
                
                self.images[sample_id] = torch.from_numpy(image).float()
                self.masks[sample_id] = torch.from_numpy(mask).long()
            
            print("Dataset preloaded into RAM")
    
    def __len__(self):
        return len(self.sample_ids)
    
    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        
        if self.preload:
            image = self.images[sample_id]
            mask = self.masks[sample_id]
        else:
            image = np.load(os.path.join(self.image_dir, f'{sample_id}.npy'))
            mask = np.load(os.path.join(self.mask_dir, f'{sample_id}.npy'))
            
            if len(image.shape) == 2:
                image = np.stack([image, image, image], axis=0)
            else:
                image = np.transpose(image, (2, 0, 1))
                if image.shape[0] == 1:
                    image = np.repeat(image, 3, axis=0)
            
            if len(mask.shape) == 2:
                mask = mask
            else:
                mask = mask[..., 0]
            
            image = torch.from_numpy(image).float()
            mask = torch.from_numpy(mask).long()
        
        return {
            'image': image,
            'mask': mask,
            'id': sample_id
        }


class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred = pred.contiguous().view(-1)
        target = target.contiguous().view(-1)
        
        intersection = (pred * target).sum()
        dice = (2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)
        
        return 1 - dice


def calculate_dice_score(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    
    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)
    
    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    
    return dice.item()


class DeepLabV3Segmentation:
    def __init__(self, num_classes=2, device='cuda', learning_rate=1e-4):
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.num_classes = num_classes
        
        self.model = deeplabv3_resnet50(weights='DEFAULT')
        self.model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
        self.model.aux_classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
        self.model = self.model.to(self.device)
        
        self.criterion = DiceLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)
        
        os.makedirs('checkpoints', exist_ok=True)
        os.makedirs('metrics', exist_ok=True)
    
    def train_epoch(self, dataloader):
        self.model.train()
        epoch_loss = 0.0
        epoch_dice = 0.0
        num_batches = 0
        
        progress_bar = tqdm(dataloader, desc='Training')
        
        for batch in progress_bar:
            images = batch['image'].to(self.device)
            masks = batch['mask'].to(self.device)
            
            self.optimizer.zero_grad()
            
            outputs = self.model(images)
            logits = outputs['out']
            
            logits_class1 = logits[:, 1:2, :, :]
            
            loss = self.criterion(logits_class1, masks.float())
            
            loss.backward()
            self.optimizer.step()
            
            dice = calculate_dice_score(logits_class1, masks.float())
            
            epoch_loss += loss.item()
            epoch_dice += dice
            num_batches += 1
            
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'dice': f'{dice:.4f}'
            })
        
        return epoch_loss / num_batches, epoch_dice / num_batches
    
    def validate(self, dataloader):
        self.model.eval()
        val_loss = 0.0
        val_dice = 0.0
        num_batches = 0
        
        with torch.no_grad():
            progress_bar = tqdm(dataloader, desc='Validation')
            
            for batch in progress_bar:
                images = batch['image'].to(self.device)
                masks = batch['mask'].to(self.device)
                
                outputs = self.model(images)
                logits = outputs['out']
                
                logits_class1 = logits[:, 1:2, :, :]
                
                loss = self.criterion(logits_class1, masks.float())
                dice = calculate_dice_score(logits_class1, masks.float())
                
                val_loss += loss.item()
                val_dice += dice
                num_batches += 1
                
                progress_bar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'dice': f'{dice:.4f}'
                })
        
        return val_loss / num_batches, val_dice / num_batches
    
    def save_checkpoint(self, epoch, train_loss, train_dice, val_loss, val_dice):
        checkpoint_path = os.path.join('checkpoints', f'deeplabv3_epoch_{epoch}.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'train_loss': train_loss,
            'train_dice': train_dice,
            'val_loss': val_loss,
            'val_dice': val_dice
        }, checkpoint_path)
        
        return checkpoint_path
    
    def save_metrics(self, epoch, train_loss, train_dice, val_loss, val_dice, checkpoint_path):
        csv_path = os.path.join('metrics', f'epoch_{epoch}_metrics.csv')
        
        with open(csv_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Metric', 'Value'])
            writer.writerow(['Epoch', epoch])
            writer.writerow(['Train Loss', train_loss])
            writer.writerow(['Train DICE', train_dice])
            writer.writerow(['Validation Loss', val_loss])
            writer.writerow(['Validation DICE', val_dice])
            writer.writerow(['Checkpoint Path', checkpoint_path])
            writer.writerow(['Model', 'DeepLabV3-ResNet50'])
            writer.writerow(['Loss Function', 'Dice Loss'])
            writer.writerow(['Optimizer', 'Adam'])
        
        return csv_path
    
    def train(self, train_loader, val_loader, num_epochs):
        print(f"Training on device: {self.device}")
        print(f"Model: DeepLabV3-ResNet50")
        print(f"Loss: Dice Loss")
        print(f"Optimizer: Adam")
        print(f"Number of epochs: {num_epochs}")
        print("-" * 50)
        
        for epoch in range(1, num_epochs + 1):
            print(f"\nEpoch {epoch}/{num_epochs}")
            
            train_loss, train_dice = self.train_epoch(train_loader)
            val_loss, val_dice = self.validate(val_loader)
            
            print(f"\nEpoch {epoch} Summary:")
            print(f"Train Loss: {train_loss:.4f}, Train DICE: {train_dice:.4f}")
            print(f"Val Loss: {val_loss:.4f}, Val DICE: {val_dice:.4f}")
            
            checkpoint_path = self.save_checkpoint(epoch, train_loss, train_dice, val_loss, val_dice)
            csv_path = self.save_metrics(epoch, train_loss, train_dice, val_loss, val_dice, checkpoint_path)
            
            print(f"Saved checkpoint: {checkpoint_path}")
            print(f"Saved metrics: {csv_path}")
            print("-" * 50)


dataset = LIDCSliceDataset('lidc_processed', preload=True)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

segmentation = DeepLabV3Segmentation(num_classes=2, device='cuda', learning_rate=1e-4)

segmentation.train(train_loader, val_loader, num_epochs=50)

In [ ]:
!ls

In [ ]:
!rm -rf checkpoints/

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision.models.segmentation import deeplabv3_resnet50
import torch.nn as nn
import os
from torch.utils.data import Dataset


class LIDCSliceDataset(Dataset):
    
    def __init__(self, data_dir='lidc_processed', transform=None, preload=False):
        self.data_dir = data_dir
        self.transform = transform
        self.preload = preload
        
        self.image_dir = os.path.join(data_dir, 'images')
        self.mask_dir = os.path.join(data_dir, 'masks')
        
        self.sample_ids = [
            f.replace('.npy', '') 
            for f in os.listdir(self.image_dir) 
            if f.endswith('.npy')
        ]
    
    def __len__(self):
        return len(self.sample_ids)
    
    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        
        image = np.load(os.path.join(self.image_dir, f'{sample_id}.npy'))
        mask = np.load(os.path.join(self.mask_dir, f'{sample_id}.npy'))
        
        if len(image.shape) == 2:
            image = np.stack([image, image, image], axis=0)
        else:
            image = np.transpose(image, (2, 0, 1))
            if image.shape[0] == 1:
                image = np.repeat(image, 3, axis=0)
        
        if len(mask.shape) == 2:
            mask = mask
        else:
            mask = mask[..., 0]
        
        image = torch.from_numpy(image).float()
        mask = torch.from_numpy(mask).long()
        
        return {
            'image': image,
            'mask': mask,
            'id': sample_id
        }


def load_latest_checkpoint(checkpoint_dir='checkpoints'):
    checkpoints = [f for f in os.listdir(checkpoint_dir) if f.endswith('.pth')]
    checkpoints.sort(key=lambda x: int(x.split('_')[-1].replace('.pth', '')))
    latest_checkpoint = checkpoints[-1]
    
    checkpoint_path = os.path.join(checkpoint_dir, latest_checkpoint)
    print(f"Loading checkpoint: {checkpoint_path}")
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    model = deeplabv3_resnet50(weights='DEFAULT')
    model.classifier[4] = nn.Conv2d(256, 2, kernel_size=1)
    model.aux_classifier[4] = nn.Conv2d(256, 2, kernel_size=1)
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    print(f"Loaded model from epoch {checkpoint['epoch']}")
    print(f"Validation DICE: {checkpoint['val_dice']:.4f}")
    
    return model, device


def predict_and_visualize(model, dataset, device, idx=0):
    sample = dataset[idx]
    
    image_tensor = sample['image'].unsqueeze(0).to(device)
    true_mask = sample['mask'].numpy()
    
    with torch.no_grad():
        output = model(image_tensor)
        logits = output['out']
        pred_mask = torch.sigmoid(logits[0, 1, :, :])
        pred_mask = (pred_mask > 0.5).float().cpu().numpy()
    
    original_image = image_tensor[0, 0, :, :].cpu().numpy()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(original_image, cmap='gray')
    axes[0].set_title('Original CT Slice')
    axes[0].axis('off')
    
    axes[1].imshow(original_image, cmap='gray')
    axes[1].imshow(true_mask, cmap='Reds', alpha=0.5)
    axes[1].set_title('Ground Truth Segmentation')
    axes[1].axis('off')
    
    axes[2].imshow(original_image, cmap='gray')
    axes[2].imshow(pred_mask, cmap='Greens', alpha=0.5)
    axes[2].set_title('AI Predicted Segmentation')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'prediction_{sample["id"]}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Sample ID: {sample['id']}")
    print(f"Prediction saved as: prediction_{sample['id']}.png")


dataset = LIDCSliceDataset('lidc_processed', preload=False)

model, device = load_latest_checkpoint('checkpoints')

predict_and_visualize(model, dataset, device, idx=100)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision.models.segmentation import deeplabv3_resnet50
import torch.nn as nn
import os
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import csv


class LIDCSliceDataset(Dataset):
    
    def __init__(self, data_dir='lidc_processed', transform=None, preload=False):
        self.data_dir = data_dir
        self.transform = transform
        self.preload = preload
        
        self.image_dir = os.path.join(data_dir, 'images')
        self.mask_dir = os.path.join(data_dir, 'masks')
        
        self.sample_ids = [
            f.replace('.npy', '') 
            for f in os.listdir(self.image_dir) 
            if f.endswith('.npy')
        ]
    
    def __len__(self):
        return len(self.sample_ids)
    
    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        
        image = np.load(os.path.join(self.image_dir, f'{sample_id}.npy'))
        mask = np.load(os.path.join(self.mask_dir, f'{sample_id}.npy'))
        
        if len(image.shape) == 2:
            image = np.stack([image, image, image], axis=0)
        else:
            image = np.transpose(image, (2, 0, 1))
            if image.shape[0] == 1:
                image = np.repeat(image, 3, axis=0)
        
        if len(mask.shape) == 2:
            mask = mask
        else:
            mask = mask[..., 0]
        
        image = torch.from_numpy(image).float()
        mask = torch.from_numpy(mask).long()
        
        return {
            'image': image,
            'mask': mask,
            'id': sample_id
        }


def calculate_dice_score(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    
    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)
    
    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    
    return dice.item()


def calculate_iou(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    
    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)
    
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    iou = (intersection + smooth) / (union + smooth)
    
    return iou.item()


def calculate_pixel_accuracy(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    
    correct = (pred == target).float().sum()
    total = target.numel()
    
    return (correct / total).item()


def load_latest_checkpoint(checkpoint_dir='checkpoints'):
    checkpoints = [f for f in os.listdir(checkpoint_dir) if f.endswith('.pth')]
    checkpoints.sort(key=lambda x: int(x.split('_')[-1].replace('.pth', '')))
    latest_checkpoint = checkpoints[-1]
    
    checkpoint_path = os.path.join(checkpoint_dir, latest_checkpoint)
    print(f"Loading checkpoint: {checkpoint_path}")
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    model = deeplabv3_resnet50(weights='DEFAULT')
    model.classifier[4] = nn.Conv2d(256, 2, kernel_size=1)
    model.aux_classifier[4] = nn.Conv2d(256, 2, kernel_size=1)
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    print(f"Loaded model from epoch {checkpoint['epoch']}")
    print(f"Training DICE: {checkpoint['train_dice']:.4f}")
    print(f"Validation DICE: {checkpoint['val_dice']:.4f}")
    
    return model, device, checkpoint['epoch']


def test_on_validation_set(model, val_loader, device):
    model.eval()
    
    all_dice = []
    all_iou = []
    all_accuracy = []
    
    with torch.no_grad():
        progress_bar = tqdm(val_loader, desc='Testing on validation set')
        
        for batch in progress_bar:
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            
            outputs = model(images)
            logits = outputs['out']
            logits_class1 = logits[:, 1:2, :, :]
            
            for i in range(logits_class1.size(0)):
                pred = logits_class1[i:i+1]
                target = masks[i:i+1].float()
                
                dice = calculate_dice_score(pred, target)
                iou = calculate_iou(pred, target)
                acc = calculate_pixel_accuracy(pred, target)
                
                all_dice.append(dice)
                all_iou.append(iou)
                all_accuracy.append(acc)
            
            progress_bar.set_postfix({
                'DICE': f'{np.mean(all_dice):.4f}',
                'IoU': f'{np.mean(all_iou):.4f}',
                'Acc': f'{np.mean(all_accuracy):.4f}'
            })
    
    return all_dice, all_iou, all_accuracy


def visualize_predictions(model, val_dataset, device, num_samples=5):
    model.eval()
    
    indices = np.random.choice(len(val_dataset), num_samples, replace=False)
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    with torch.no_grad():
        for idx, sample_idx in enumerate(indices):
            sample = val_dataset[sample_idx]
            
            image_tensor = sample['image'].unsqueeze(0).to(device)
            true_mask = sample['mask'].numpy()
            
            output = model(image_tensor)
            logits = output['out']
            pred_mask = torch.sigmoid(logits[0, 1, :, :])
            pred_mask = (pred_mask > 0.5).float().cpu().numpy()
            
            original_image = image_tensor[0, 0, :, :].cpu().numpy()
            
            axes[idx, 0].imshow(original_image, cmap='gray')
            axes[idx, 0].set_title(f'Original CT Slice\n{sample["id"]}')
            axes[idx, 0].axis('off')
            
            axes[idx, 1].imshow(original_image, cmap='gray')
            axes[idx, 1].imshow(true_mask, cmap='Reds', alpha=0.5)
            axes[idx, 1].set_title('Ground Truth')
            axes[idx, 1].axis('off')
            
            axes[idx, 2].imshow(original_image, cmap='gray')
            axes[idx, 2].imshow(pred_mask, cmap='Greens', alpha=0.5)
            axes[idx, 2].set_title('AI Prediction')
            axes[idx, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig('validation_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("Visualization saved as: validation_predictions.png")


def save_test_results(epoch, dice_scores, iou_scores, accuracy_scores):
    os.makedirs('test_results', exist_ok=True)
    
    csv_path = os.path.join('test_results', f'validation_test_epoch_{epoch}.csv')
    
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Metric', 'Mean', 'Std', 'Min', 'Max', 'Median'])
        writer.writerow(['DICE Score', 
                        np.mean(dice_scores), 
                        np.std(dice_scores),
                        np.min(dice_scores),
                        np.max(dice_scores),
                        np.median(dice_scores)])
        writer.writerow(['IoU Score', 
                        np.mean(iou_scores), 
                        np.std(iou_scores),
                        np.min(iou_scores),
                        np.max(iou_scores),
                        np.median(iou_scores)])
        writer.writerow(['Pixel Accuracy', 
                        np.mean(accuracy_scores), 
                        np.std(accuracy_scores),
                        np.min(accuracy_scores),
                        np.max(accuracy_scores),
                        np.median(accuracy_scores)])
    
    print(f"\nTest results saved to: {csv_path}")
    
    detailed_csv_path = os.path.join('test_results', f'validation_detailed_epoch_{epoch}.csv')
    
    with open(detailed_csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Sample_Index', 'DICE', 'IoU', 'Accuracy'])
        for i, (dice, iou, acc) in enumerate(zip(dice_scores, iou_scores, accuracy_scores)):
            writer.writerow([i, dice, iou, acc])
    
    print(f"Detailed results saved to: {detailed_csv_path}")


dataset = LIDCSliceDataset('lidc_processed', preload=False)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

print(f"Validation samples: {len(val_dataset)}")

model, device, epoch = load_latest_checkpoint('checkpoints')

print("\nTesting model on validation dataset...")
dice_scores, iou_scores, accuracy_scores = test_on_validation_set(model, val_loader, device)

print("\n" + "="*50)
print("VALIDATION SET TEST RESULTS")
print("="*50)
print(f"Mean DICE Score: {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}")
print(f"Median DICE Score: {np.median(dice_scores):.4f}")
print(f"Min/Max DICE Score: {np.min(dice_scores):.4f} / {np.max(dice_scores):.4f}")
print(f"\nMean IoU Score: {np.mean(iou_scores):.4f} ± {np.std(iou_scores):.4f}")
print(f"Median IoU Score: {np.median(iou_scores):.4f}")
print(f"Min/Max IoU Score: {np.min(iou_scores):.4f} / {np.max(iou_scores):.4f}")
print(f"\nMean Pixel Accuracy: {np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}")
print(f"Median Pixel Accuracy: {np.median(accuracy_scores):.4f}")
print(f"Min/Max Pixel Accuracy: {np.min(accuracy_scores):.4f} / {np.max(accuracy_scores):.4f}")
print("="*50)

save_test_results(epoch, dice_scores, iou_scores, accuracy_scores)

print("\nGenerating sample predictions...")
visualize_predictions(model, val_dataset, device, num_samples=5)